# 综合应用 (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]

🤗Transformers API 能够通过一个**高级函数**为我们处理所有这些工作（分词、转换为 inputs ID、填充、截断以及注意力掩码），接下来我们就要深入研究这个函数。

当你直接在句子上调用你的 `tokenizer `时，就可以得到转换后的可以直接放入模型的数据了：



In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

model_inputs = tokenizer(sequence)

`model_inputs` 变量包含**模型运行所需的所有数据**。

在 DistilBERT 中，它包括 inputs ID 和注意力掩码。其他接受额外输入的模型也会有对应的 tokenizer

这个函数非常强大。首先，它可以对单个句子进行处理：

In [ ]:
sequence = "I've been waiting for a HuggingFace course my whole life."

model_inputs = tokenizer(sequence)

它还一次处理多个多个，并且 API 的用法完全一致：

In [ ]:
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

model_inputs = tokenizer(sequences)

它可以使用多种不同的方法对目标进行填充：

In [ ]:
# 将句子序列填充到最长句子的长度
model_inputs = tokenizer(sequences, padding="longest")

# 将句子序列填充到模型的最大长度
# (512 for BERT or DistilBERT)
model_inputs = tokenizer(sequences, padding="max_length")

# 将句子序列填充到指定的最大长度
model_inputs = tokenizer(sequences, padding="max_length", max_length=8)

它还可以对序列进行截断：

In [ ]:
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

# 将截断比模型最大长度长的句子序列
# (512 for BERT or DistilBERT)
model_inputs = tokenizer(sequences, truncation=True)

# 将截断长于指定最大长度的句子序列
model_inputs = tokenizer(sequences, max_length=8, truncation=True)

##特殊的 tokens

如果我们看一下 tokenizer 返回的 inputs ID，我们会发现它们与之前的略有不同：

In [ ]:
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

# Returns PyTorch tensors
model_inputs = tokenizer(sequences, padding=True, return_tensors="pt")

# Returns TensorFlow tensors
model_inputs = tokenizer(sequences, padding=True, return_tensors="tf")

# Returns NumPy arrays
model_inputs = tokenizer(sequences, padding=True, return_tensors="np")

In [ ]:
sequence = "I've been waiting for a HuggingFace course my whole life."

model_inputs = tokenizer(sequence)
print(model_inputs["input_ids"])

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012, 102]
[1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012]

In [ ]:
print(tokenizer.decode(model_inputs["input_ids"]))
print(tokenizer.decode(ids))

"[CLS] i've been waiting for a huggingface course my whole life. [SEP]"
"i've been waiting for a huggingface course my whole life."

okenizer 在开头添加了特殊单词 `[CLS]` ，在结尾添加了特殊单词 `[SEP] `。这是因为模型在预训练时使用了这些字词，所以为了得到相同的推断结果，我们也需要添加它们。

有些模型不添加特殊单词，或者添加不同的特殊单词；模无论如何，`tokenizer `知道哪些是必需的，并会为你处理这些问题。

##小结：从 tokenizer 到模型

tokens 是什么
tokenizer(...) 返回一个字典，结构形如：




```
"input_ids": tensor, "attention_mask": tensor
```


`**tokens`： 字典解包
`** `把字典里所有键值对拆开，等价于：

```
output = model(input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"])
```

也就是把模型需要的所有输入参数传给模型。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

tokens = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")
output = model(**tokens)